In [1]:
from pyspark.sql import SparkSession # Untuk mengimpor SparkSession dari PySpark

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \ # Tempat untuk membuat SparkSession
    .appName("Pertemuan4-PengenalanPySpark") \ # memberikan nama aplikasi Spark: Pertemuan4-PengenalanSpark
    .master("local[*]") \ # spark dijalankan dalam local mode dan menggunakan core CPU
    .getOrCreate() # buat spparksession baru jika belu ada/ gunakan sparksession yang sudah ada

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR") # mengatur level pesaan log Spark meNJaDI ERROR 

print("SparkSession berhasil dibuat!") # ini hanya digunakan untuk mengecek bahwa sparksesion berhasil dibuat dan meliaht versi.
print("Versi Spark:", spark.version)

26/09/16 20:19:13 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/16 20:19:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 20:19:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [1]:
from pyspark.sql import SparkSession # Untuk mengimpor SparkSession dari PySpark

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \ # Tempat untuk membuat SparkSession
    .appName("Pertemuan4-PengenalanPySpark") \ # memberikan nama aplikasi Spark: Pertemuan4-PengenalanSpark
    .master("local[*]") \ # spark dijalankan dalam local mode dan menggunakan core CPU
    .getOrCreate() # buat spparksession baru jika belu ada/ gunakan sparksession yang sudah ada

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR") # mengatur level pesaan log Spark meNJaDI ERROR 

print("SparkSession berhasil dibuat!") # ini hanya digunakan untuk mengecek bahwa sparksesion berhasil dibuat dan meliaht versi.
print("Versi Spark:", spark.version)

26/09/16 20:19:13 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/16 20:19:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 20:19:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [3]:
# Membuat ulang dataset contoh (identik dengan Tugas Mandiri Pertemuan 2) jika belum ada
import os
if not os.path.exists("data_transaksi_ecommerce.csv"): # sistem memeriksa si file sudah ada atau belum
    import numpy as np
    import pandas as pd
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")
    data = {
        "order_id": [f"ORD-{1000+i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25,0.25,0.20,0.15,0.15]),
        "kota": np.random.choice(kota_list, size=n),
        "unit_terjual": np.random.randint(1, 10, size=n),
        "harga_satuan": np.random.choice([25000,50000,75000,100000,150000,250000,500000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35,0.30,0.20,0.15]),
    }
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False) # nah kalau data belum ada, sistem akan otomatis membuat datanya.
    print("Dataset dibuat ulang.")
else:
    print("Dataset sudah tersedia.") # kalau sudah tersedia akan langsung muncul ini

Dataset sudah tersedia.


In [2]:
# Membaca berkas CSV lokal menjadi Spark DataFrame
# header=True    -> baris pertama dianggap nama kolom
# inferSchema=True -> Spark otomatis menebak tipe data tiap kolom (angka, teks, dst.)
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True) # Mengubah file teks CVS menjadi Spark DataFrame.
# header=True untuk memberatu spark baris paling atas file CVS adalah nama kolom
# inferSchema=True untuk menyuruh spark otomatis mendeteksi tipe data

print("Tipe objek:", type(df)) # mencetak tipe data dari variabel df
df.printSchema() # untuk melihat kerangka data.

Tipe objek: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)



Perhatikan `printSchema()` menampilkan struktur kolom beserta tipe datanya — sangat berguna untuk memverifikasi Spark membaca data dengan benar, terutama pada dataset besar yang tidak mungkin dilihat seluruhnya sekaligus.

In [3]:
# Menampilkan beberapa baris pertama — mirip df.head() di pandas, namun disebut show()
df.show(5)

# Menghitung jumlah baris — mirip len(df) di pandas
print("Jumlah baris:", df.count()) # menghitung jumlah seluruh baris yang ada di DataFrame

+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|          Elektronik|      Solo|           3|       50000|         E-Wallet|
|ORD-1004|2026-07-11 00:00:00|          Elektronik|  Semarang|           5|      250000|    Transfer Bank|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
only showing top 5 rows

Jumlah baris

---
## 4.7 Operasi Dasar DataFrame: Select, Filter, GroupBy, Agg

### 4.7.1 `select()` — Memilih Kolom Tertentu

In [4]:
df.select("order_id", "kota", "kategori").show(5)

+--------+----------+--------------------+
|order_id|      kota|            kategori|
+--------+----------+--------------------+
|ORD-1000|  Magelang|Kesehatan & Kecan...|
|ORD-1001|Yogyakarta|   Makanan & Minuman|
|ORD-1002| Purworejo|             Fashion|
|ORD-1003|      Solo|          Elektronik|
|ORD-1004|  Semarang|          Elektronik|
+--------+----------+--------------------+
only showing top 5 rows



### 4.7.2 `filter()` — Menyaring Baris

In [5]:
from pyspark.sql.functions import col

# Menyaring transaksi kategori Elektronik dengan unit_terjual lebih dari 5
df.filter((col("kategori") == "Elektronik") & (col("unit_terjual") > 5)).show(5)

+--------+-------------------+----------+----------+------------+------------+-----------------+
|order_id|            tanggal|  kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+----------+----------+------------+------------+-----------------+
|ORD-1019|2026-07-03 00:00:00|Elektronik|  Magelang|           9|       75000|         E-Wallet|
|ORD-1020|2026-07-22 00:00:00|Elektronik|  Semarang|           6|      150000|     Kartu Kredit|
|ORD-1028|2026-07-28 00:00:00|Elektronik|  Semarang|           8|      250000|    Transfer Bank|
|ORD-1040|2026-07-31 00:00:00|Elektronik| Purworejo|           6|      100000|         E-Wallet|
|ORD-1054|2026-07-25 00:00:00|Elektronik|Yogyakarta|           9|      500000|    Transfer Bank|
+--------+-------------------+----------+----------+------------+------------+-----------------+
only showing top 5 rows



> Fungsi `col("nama_kolom")` digunakan untuk merujuk sebuah kolom secara eksplisit — praktik standar di PySpark, terutama saat menuliskan kondisi kompleks seperti di atas.

### 4.7.3 `groupBy()` dan `agg()` — Meringkas Data

In [9]:
from pyspark.sql.functions import sum as spark_sum, count, avg

# Menambahkan kolom baru: total_pendapatan = unit_terjual x harga_satuan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Meringkas: total pendapatan & jumlah transaksi per kota, diurutkan dari tertinggi
ringkasan_kota = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan"),
    count("order_id").alias("jumlah_transaksi"),
    avg("unit_terjual").alias("rata_rata_unit")
).orderBy(col("total_pendapatan").desc())

ringkasan_kota.show()

+----------+----------------+----------------+-----------------+
|      kota|total_pendapatan|jumlah_transaksi|   rata_rata_unit|
+----------+----------------+----------------+-----------------+
|  Magelang|       109925000|             119| 5.07563025210084|
| Purworejo|       104675000|             122|4.737704918032787|
|Yogyakarta|       104625000|             129|5.248062015503876|
|  Semarang|        87200000|             118|4.771186440677966|
|      Solo|        83950000|             112|5.053571428571429|
+----------+----------------+----------------+-----------------+



In [ ]:
4.8 Bonus: Membaca Data Langsung dari HDFS¶

In [10]:
# Upload dataset ke HDFS terlebih dahulu (jika belum ada dari Pertemuan 3)
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan4
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/mahasiswa/pertemuan4/

# Membaca CSV LANGSUNG dari HDFS menggunakan Spark — perhatikan prefix "hdfs://"
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv",
    header=True, inferSchema=True
)
print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(5)

Jumlah baris dari HDFS: 600
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|          Elektronik|      Solo|           3|       50000|         E-Wallet|
|ORD-1004|2026-07-11 00:00:00|          Elektronik|  Semarang|           5|      250000|    Transfer Bank|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
only show


---

## Menutup SparkSession

Kebiasaan baik: selalu tutup SparkSession di akhir notebook untuk clear up resources (memori & CPU) yang dipakainya.

In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


> Jika ingin melanjutkan latihan setelah ini, jalankan ulang cell pembuatan `SparkSession` pada Sub-bab 4.5.


---
## Latihan Mandiri

Buat SparkSession baru terlebih dahulu (copy cell dari Sub-bab 4.5) sebelum mengerjakan latihan berikut.

In [12]:
# Jalankan ini dulu sebelum mengerjakan latihan di bawah
spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

Siap. Jumlah baris: 600


**Soal 1.** Tampilkan hanya kolom `order_id`, `kategori`, dan `total_pendapatan` untuk transaksi dengan `metode_pembayaran` bernilai `"E-Wallet"`.

In [12]:
# Jawaban Soal 1 di sini
df.filter(col("metode_pembayaran") == "E-Wallet") \
    .select("order_id", "kategori", "total_pendapatan") \
    .show()

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
|ORD-1015|          Elektronik|          375000|
|ORD-1017|Kesehatan & Kecan...|         4000000|
|ORD-1019|          Elektronik|          675000|
|ORD-1021|          Elektronik|          200000|
|ORD-1022|        Rumah Tangga|          900000|
|ORD-1023|Kesehatan & Kecan...|         2500000|
|ORD-1024|             Fashion|          750000|
|ORD-1027|             Fashion|           50000|
|ORD-1029|             Fashion|          600000|
|ORD-1030|             Fashion|          300000|
|ORD-1033|          Elektronik|          300000|
|ORD-1035|   Makanan & Minuman|         2500000|
|ORD-1036|   Makanan

In [12]:
# Jawaban Soal 1 di sini
df.filter(col("metode_pembayaran") == "E-Wallet") \
    .select("order_id", "kategori", "total_pendapatan") \
    .show()

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
|ORD-1015|          Elektronik|          375000|
|ORD-1017|Kesehatan & Kecan...|         4000000|
|ORD-1019|          Elektronik|          675000|
|ORD-1021|          Elektronik|          200000|
|ORD-1022|        Rumah Tangga|          900000|
|ORD-1023|Kesehatan & Kecan...|         2500000|
|ORD-1024|             Fashion|          750000|
|ORD-1027|             Fashion|           50000|
|ORD-1029|             Fashion|          600000|
|ORD-1030|             Fashion|          300000|
|ORD-1033|          Elektronik|          300000|
|ORD-1035|   Makanan & Minuman|         2500000|
|ORD-1036|   Makanan

In [12]:
# Jawaban Soal 1 di sini
df.filter(col("metode_pembayaran") == "E-Wallet") \
    .select("order_id", "kategori", "total_pendapatan") \
    .show()

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
|ORD-1015|          Elektronik|          375000|
|ORD-1017|Kesehatan & Kecan...|         4000000|
|ORD-1019|          Elektronik|          675000|
|ORD-1021|          Elektronik|          200000|
|ORD-1022|        Rumah Tangga|          900000|
|ORD-1023|Kesehatan & Kecan...|         2500000|
|ORD-1024|             Fashion|          750000|
|ORD-1027|             Fashion|           50000|
|ORD-1029|             Fashion|          600000|
|ORD-1030|             Fashion|          300000|
|ORD-1033|          Elektronik|          300000|
|ORD-1035|   Makanan & Minuman|         2500000|
|ORD-1036|   Makanan

**Soal 2.** Hitung total pendapatan **per kategori** (bukan per kota), urutkan dari yang tertinggi.

In [13]:
# Jawaban Soal 2 di sini
df.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show()


+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|             Fashion|       124825000|
|          Elektronik|       110600000|
|   Makanan & Minuman|        95400000|
|Kesehatan & Kecan...|        82200000|
|        Rumah Tangga|        77350000|
+--------------------+----------------+



**Soal 3.** Tampilkan jumlah transaksi untuk masing-masing `metode_pembayaran` (tidak pakai `agg`).

In [14]:
# Jawaban Soal 3 di sini
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  114|
|    Transfer Bank|  208|
|     Kartu Kredit|   85|
|         E-Wallet|  193|
+-----------------+-----+



**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: apa yang dimaksud dengan *lazy evaluation* di PySpark, dan mengapa hal ini menguntungkan ketika bekerja dengan data berskala besar? Tulis jawaban pada markdown cell di bawah ini.

Lazy Evaluation merupakan spark yang tidak langsung mengesekusi perintah (filter atau seleksi) sampai ada perintah aksi yang benar-benar meminta hasil(.show atau .count()). Hal itu memberi keuntungan karena saat ada data besar, data tersebut dapat diproses dulu dengan cara yang paling efisien dan enghindari proses yang tidak diperlukan.

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

>  **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 5 dimulai**.
>  **Sifat tugas:** individu.

### Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [15]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


### Instruksi Pengerjaan

Buat notebook baru **`Tugas4_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` baru, lalu kerjakan bagian **A sampai E** berikut — **seluruhnya wajib menggunakan PySpark, bukan pandas**, dan data **wajib dibaca langsung dari HDFS** (`hdfs://localhost:9000/...`), bukan dari berkas lokal.

---

**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).

**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya 

**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

> **Catatan:** Spark menyimpan hasil sebagai **beberapa berkas partisi** (`part-00000...`, dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark. Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas4_[NIM]_[Nama Lengkap].ipynb` melalui ELITA, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- **Dilarang menggunakan pandas** untuk bagian analisis A-E (boleh dipakai hanya di sel pembuatan dataset yang sudah disediakan).

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Baca & Eksplorasi | Data berhasil dibaca dari HDFS, struktur & jumlah baris benar | 15% |
| B. Data Kosong | Missing value teridentifikasi & ditangani dengan alasan yang logis | 15% |
| C. Transformasi | Kedua kolom baru dihitung dengan benar menggunakan fungsi PySpark yang tepat | 20% |
| D. Analisis GroupBy | Ketiga jawaban analitis benar & menggunakan sintaks PySpark (bukan pandas) | 30% |
| E. Simpan ke HDFS | Hasil berhasil tersimpan ke HDFS; penjelasan tentang partisi tepat | 20% |


In [19]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [20]:
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [21]:
from pyspark.sql.functions import col

jumlah_rating_kosong = df.filter(col("rating").isNull()).count()

print("Jumlah rating kosong:", jumlah_rating_kosong)

Jumlah rating kosong: 204


In [22]:
df = df.na.drop(subset=["rating"])

print("Jumlah baris setelah missing value ditangani:", df.count())

Jumlah baris setelah missing value ditangani: 796


Saya memilih df.na.drop(subset=["rating"]) karena nilai rating yang kosong tidak dapat ditentukan secara akurat. Daripada mengisi dengan nilai tertentu yang dapat memengaruhi hasil rata-rata rating, data dengan rating kosong dihapus sehingga analisis rating hanya menggunakan data yang memiliki nilai rating.

In [23]:
from pyspark.sql.functions import when

df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar")
    .otherwise("Kecil")
)

df.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [24]:
df.groupBy("kategori") \
    .sum("total_pendapatan") \
    .withColumnRenamed("sum(total_pendapatan)", "total_pendapatan") \
    .orderBy(col("total_pendapatan").desc()) \
    .show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       108285000|
|   Makanan & Minuman|       106085000|
|             Fashion|       101725000|
|Kesehatan & Kecan...|        98040000|
|            Olahraga|        94230000|
|          Elektronik|        89325000|
+--------------------+----------------+



In [25]:
df.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   74|
|   Kebumen|   63|
|Yogyakarta|   61|
| Purworejo|   55|
|  Magelang|   52|
|  Semarang|   47|
+----------+-----+



In [26]:
from pyspark.sql.functions import avg

df.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show()

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|         E-Wallet|4.135678391959799|
|     Kartu Kredit|4.109947643979058|
+-----------------+-----------------+



In [27]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Data berhasil disimpan ke HDFS.")

Data berhasil disimpan ke HDFS.


In [28]:
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi

Found 2 items
-rw-r--r--   3 vboxuser supergroup          0 2026-09-16 23:16 /user/mahasiswa/tugas4/hasil_transaksi/_SUCCESS
-rw-r--r--   3 vboxuser supergroup      77337 2026-09-16 23:16 /user/mahasiswa/tugas4/hasil_transaksi/part-00000-01709b4b-8f55-4814-a77d-8e850b2d5117-c000.csv


Spark menghasilkan file part-00000...csv karena Spark memproses data menggunakan sistem partisi. Data dapat dibagi menjadi beberapa partisi dan setiap partisi dapat menghasilkan file sendiri. Berbeda dengan pandas yang biasanya menyimpan hasil ke dalam satu file CSV, Spark menggunakan beberapa file part sebagai bagian dari proses pemrosesan data terdistribusi.